In [2]:
from langchain_text_splitters import TokenTextSplitter
from typing import List

import sys
import os

# Define paths dynamically
base_dir_str: str = os.path.abspath("")  
data_rel_path_str: str = os.path.join("..", "data") 
data_abs_path_str: str = os.path.normpath(os.path.join(base_dir_str, data_rel_path_str))

# Add the absolute path to sys.path to ensure module imports work correctly
if data_abs_path_str not in sys.path:
    sys.path.append(data_abs_path_str)

# Import your custom dataset
try:
    import dataset
    dataset_imported_bol: bool = True
except ImportError as e:
    print(f"Warning: Could not import dataset module from {data_abs_path_str}. Error: {e}")
    dataset_imported_bol: bool = False


def get_sliding_window_chunks_list(
    input_str: str,
    chunk_size_int: int = 200,
    overlap_size_int: int = 50
) -> List[str]:
    """
    Splits a raw string into overlapping chunks using a sliding window approach based on tokens.

    Args:
        input_str (str): The raw text data to be processed.
        chunk_size_int (int): The total number of tokens in each window (chunk).
        overlap_size_int (int): The number of tokens from the previous window to include in the current one.

    Returns:
        List[str]: A list of overlapping text chunks.
    """
    # The 'sliding' effect is created by the overlap_size_int
    sliding_splitter_obj = TokenTextSplitter(
        chunk_size=chunk_size_int,
        chunk_overlap=overlap_size_int
    )

    chunks_list = sliding_splitter_obj.split_text(input_str)
    return chunks_list

# Example: Using a 25% overlap (50 tokens overlap for a 200 token window)
raw_document_str = dataset.test


sliding_chunks_list = get_sliding_window_chunks_list(
    input_str=raw_document_str,
    chunk_size_int=200,
    overlap_size_int=50
)

print(f"Generated {len(sliding_chunks_list)} overlapping chunks.")

c:\SJ\SD_tasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generated 35 overlapping chunks.


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def initialize_sliding_vector_db(chunks_list: List[str]) -> Chroma:
    """
    Embeds overlapping chunks and initializes a ChromaDB instance.

    Args:
        chunks_list (List[str]): The list of overlapping text chunks.

    Returns:
        Chroma: The initialized vector database object.
    """
    # Using a standard embedding model
    embedding_fn_obj = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # We use a unique collection name to differentiate from fixed chunking
    sliding_vector_db = Chroma.from_texts(
        texts=chunks_list,
        embedding=embedding_fn_obj,
        collection_name="sliding_window_rag"
    )

    return sliding_vector_db

sliding_db_obj = initialize_sliding_vector_db(sliding_chunks_list)

C:\Users\sjoshi06\AppData\Local\Temp\ipykernel_22180\3241771410.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_fn_obj = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 400.99it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def query_sliding_window_db(
    query_str: str,
    vector_db: Chroma,
    top_k_int: int = 3
) -> List[str]:
    """
    Retrieves the most relevant overlapping chunks for a given query.

    Args:
        query_str (str): The search query.
        vector_db (Chroma): The vector database containing overlapping chunks.
        top_k_int (int): Number of chunks to retrieve.

    Returns:
        List[str]: The top matching text segments.
    """
    # Performing similarity search
    search_results_list = vector_db.similarity_search(query_str, k=top_k_int)

    # Extracting text content
    final_context_list = [doc_obj.page_content for doc_obj in search_results_list]
    return final_context_list

# Testing the sliding window retrieval
search_query_str = "What is clonal hematopoiesis and how common is it in elderly individuals?"
relevant_context_list = query_sliding_window_db(search_query_str, sliding_db_obj)

for index_int, chunk_text_str in enumerate(relevant_context_list):
    print(f"Chunk {index_int + 1}: {chunk_text_str}")

Chunk 1:  in the exome was common at all ages, in contrast to the strongly age-dependent acquisition of clonal hematopoiesis with candidate drivers (Fig. S13 in the Supplementary Appendix). However, the presence of two detectable putative somatic mutations was more age-dependent, occurring in 1.3% of persons younger than 50 years of age as compared with 4.0% of those older than 65 years of age (Fig. S13 in the Supplementary Appendix). The presence of three or more putative somatic mutations (our criterion for clonal hematopoiesis with unknown drivers) was more strongly age-dependent, occurring in only 0.3% of persons younger than 50 years of age but in 4.6% of those older than 65 years of age (Figure 2D). This age trajectory is similar to that of clonal hematopoiesis with candidate drivers.
Overall, clonal hematopoiesis with candidate or
Chunk 2:  clonal hematopoiesis with unknown drivers. In some cases of clonal hematopoiesis with unknown drivers, additional analysis suggested potenti

In [5]:
# Testing the sliding window retrieval
search_query_str = "Which genes were most frequently mutated in clonal hematopoiesis?"
relevant_context_list = query_sliding_window_db(search_query_str, sliding_db_obj)

for index_int, chunk_text_str in enumerate(relevant_context_list):
    print(f"Chunk {index_int + 1}: {chunk_text_str}")

Chunk 1:  the same clone.
To consider cases of clonal hematopoiesis without obvious driver mutations, we sought to define a highly specific criterion for clonal hematopoiesis that depended on the number of mutations alone, rather than on the identity of the mutations. Of 11,845 participants with sequencing data of sufficient quality for the detection of somatic mutations, 9927 had no putative somatic mutations, 1333 had 1 mutation, 313 had 2 mutations, and 272 had 3 to 18 mutations. This distribution suggested that a random (Poisson) process (with a constant mean) could not explain the surprisingly high numbers of participants with 3 to 18 detectable mutations. In our analyses below, we classified 195 participants as having clonal hematopoiesis with unknown drivers. In some cases of clonal hematopoiesis with unknown drivers, additional analysis suggested potential candidate drivers (Fig. S12 in the Supplementary Appendix).
Clonal Hematop
Chunk 2:  the presence of a clone. Participants 